# CaMK parameter fitting

Fitting CaMKII parameters to experimental decay rates.

## Load packages

In [ ]:
using ModelingToolkit
using ModelingToolkit: t_nounits as t, D_nounits as D
using OrdinaryDiffEq
using Optimization
using OptimizationLBFGSB
using ADTypes
using ForwardDiff
using CaMKIIModel
using CaMKIIModel: second, get_camkii_simp_eqs, get_camkii_simp_sys
using Plots
using DiffEqCallbacks

## Experimental data

In [ ]:
# CaMKII activity decay times for 1Hz pacing for 15.0, 30.0, 60.0, 90.0 seconds
experimental_taus=[16.48, 16.73, 17.65, 18.08]

## Calcium transient curve

Simulated calcium transients fitted against those in the whole model with 1 Hz pacing. (for `0 <= t <= 1000ms`). See "Calcium transient fitting".

In [ ]:
@variables tau(t) Ca(t)

eqs = [
    D(tau) ~ 1, ## Implicit time variable to track time
    Ca ~ (0.2367785753064364 + -0.0006719530261280267 * tau -7.296575950651411e-6 *tau^2 + 3.466997839263819e-8 * tau^3 + 1.4262835164132568e-11 * tau^4) / (1.0 -0.01087485057558349 * tau + 5.083809280934372e-5 * tau^2 - 1.3733065253686728e-7 * tau^3 + 2.699177596317904e-10 * tau^4)  ## Fitted calcium transient
]

## ODE System

In [ ]:
eqs_camkii, CaMKAct = get_camkii_simp_eqs(;Ca = Ca)
@time "Build system" @mtkcompile sys = ODESystem([eqs_camkii; eqs], t)

In [ ]:
@unpack tau, CaMKAct, k_P1_P2, CaMKA2 = sys
ups = [tau => 0.0, k_P1_P2 => 0.0, CaMKA2 => 0.0]  ## Disable second autophosphorylation
tend = 150second
@time "Build problem" prob = ODEProblem(sys, ups, (0.0, tend))

Events to reset `tau` every second to simulate calcium transient.

In [ ]:
resettau! = (integrator) -> (integrator[tau] = 0.0)
pace15 = PresetTimeCallback(0:1second:15second, resettau!)
pace30 = PresetTimeCallback(0:1second:30second, resettau!)
pace60 = PresetTimeCallback(0:1second:60second, resettau!)
pace90 = PresetTimeCallback(0:1second:90second, resettau!)

## Test run

In [ ]:
sols = map([pace15, pace30, pace60, pace90]) do cb
    solve(prob, TRBDF2(), callback = cb)
end

plot()
for (sol, t) in zip(sols, [15.0, 30.0, 60.0, 90.0])
    plot!(sol, idxs=CaMKAct, label="Paced at $(t) seconds")
end
plot!(xlabel="Time (ms)", ylabel="CaMKII Activity", title="Simulated calcium transient", legend=:topright)

## Loss function

In [ ]:
@unpack kphos_CaMK, kdeph_CaMK, kb_CaMKP = sys
data = (
    prob=prob,
    cbs=[pace15, pace30, pace60, pace90],
    experimental_taus=experimental_taus,
    pacing_durations=[15.0, 30.0, 60.0, 90.0],
    kphos_dephos_ratio=prob.ps[kphos_CaMK] / prob.ps[kdeph_CaMK],
    a0 = sols[1](tend, idxs=CaMKAct)
)

In [ ]:
function loss(theta, data)
    @unpack prob, cbs, experimental_taus, pacing_durations, kphos_dephos_ratio, a0 = data
    @unpack kb_CaMKP, kdeph_CaMK, kphos_CaMK, CaMKAct = prob.f.sys

    ## Parallel ensemble simulation
    function prob_func(prob, i, repeat)
        remake(prob, p=[kb_CaMKP => exp10(theta[1]), kdeph_CaMK => exp10(theta[2]), kphos_CaMK => kphos_dephos_ratio * exp10(theta[2])], callback=cbs[i])
    end

    ## Calculate loss in the output function
    function output_func(sol, i)
        SciMLBase.successful_retcode(sol) || return (Inf, false)
        stimend = pacing_durations[i] * second
        ts = range(0, length=51, step=1second)
        ysim = sol(stimend .+ ts, idxs=CaMKAct).u
        peakAct = sol(stimend, idxs=CaMKAct)
        yexpected = a0 .+ (peakAct - a0) .* exp.(-ts ./ (experimental_taus[i] * second))
        error = sum(abs2, ysim .- yexpected) / length(ysim)
        return (error, false)
    end

    ensemble_prob = EnsembleProblem(prob; prob_func, output_func)
    sim = solve(ensemble_prob, TRBDF2(); trajectories=length(pacing_durations), maxiters=100000, verbose=false)
    return sum(sim)
end

Testing the loss function.

In [ ]:
theta = [log10(prob.ps[kb_CaMKP]), log10(prob.ps[kdeph_CaMK])]
@time loss(theta, data)

## Optimization

In [ ]:
optf = OptimizationFunction(loss, ADTypes.AutoForwardDiff())
optprob = OptimizationProblem(optf, theta, data, lb=[-0.5, -2] + theta, ub=[2, 2] + theta)

In [ ]:
@time sol = solve(optprob, OptimizationLBFGSB.LBFGSB())
sol.objective

In [ ]:
fit = Dict(kb_CaMKP => exp10(sol.u[1]), kdeph_CaMK => exp10(sol.u[2]), kphos_CaMK => data.kphos_dephos_ratio * exp10(sol.u[2]))

In [ ]:
println("Fitted parameters:")
println("Dissociation time of CaMKP --> CaMKA: " , 1e-3 / fit[kb_CaMKP], " seconds.")
println("Dephosphorylation time of CaMKA: " , 1e-3 / fit[kdeph_CaMK], " seconds.")
println("Phosphorylation rate of CaMKB: " , fit[kphos_CaMK] * 1000, " Hz")

## Test the fitted parameters

In [ ]:
newprob = remake(prob, p=[kb_CaMKP => fit[kb_CaMKP], kdeph_CaMK => fit[kdeph_CaMK], kphos_CaMK => fit[kphos_CaMK]])

sols = map([pace15, pace30, pace60, pace90]) do cb
    solve(newprob, TRBDF2(), callback = cb)
end

plot()
for (sol, t) in zip(sols, [15.0, 30.0, 60.0, 90.0])
    plot!(sol, idxs=CaMKAct, label="Paced at $(t) seconds")
end
plot!(xlabel="Time (ms)", ylabel="CaMKII Activity", title="Simulated calcium transient", legend=:topright)